# 12 — BRD Gap Analysis (4 hạng mục ưu tiên)

Bổ sung các phân tích còn thiếu trong BRD v1.1 (mục 3.4) trên `hotel_bookings_v5.csv`:

1. **Interaction 3 chiều** — Lead time × Segment × Mùa vụ
2. **Revenue Loss** — Doanh thu tiềm năng mất do hủy phòng
3. **Room Mis-match** — Upgrade vs Downgrade (nguyên nhân free upgrade)
4. **Deposit Simulation** — Mô phỏng chính sách cọc Online TA

Output: `reports/12_brd_gap_analysis.md`, `reports/12_brd_v1_2.md`, biểu đồ `reports/figures/12/`

In [ ]:
import os
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

%matplotlib inline
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.figsize"] = (12, 5)

NOTEBOOK_DIR = Path(os.environ.get("VSCODE_NOTEBOOK_DIR", Path.cwd()))
ROOT = NOTEBOOK_DIR.parent if (NOTEBOOK_DIR.parent / "data").is_dir() else NOTEBOOK_DIR
if str(ROOT / "notebooks") not in sys.path:
    sys.path.insert(0, str(ROOT / "notebooks"))

from _run_12_brd_gap_analysis import (
    DATA_PATH,
    FIG_DIR,
    analyze_deposit_simulation,
    analyze_interaction_3d,
    analyze_revenue_loss,
    analyze_room_mismatch,
    fmt_eur,
    fmt_int,
    fmt_pct,
    load_data,
    save_figures,
    write_brd,
    write_report,
)

df = load_data()
print(f"Đọc: {DATA_PATH.name} — {len(df):,} booking")

## 1. Interaction 3 chiều (Lead time × Segment × Mùa vụ)

Câu hỏi BRD: *Tỷ lệ hủy của Online TA có lead_time > 90 ngày vào Jul–Aug là bao nhiêu? Nhóm này chiếm bao nhiêu % tổng doanh thu bị tổn thất?*

In [ ]:
i3d = analyze_interaction_3d(df)

summary_3d = pd.DataFrame([
    {"Tổ hợp": "Online TA + lead>90 + Jul-Aug", "Booking": i3d["bookings"],
     "Tỷ lệ hủy": f"{i3d['cancel_rate']*100:.2f}%",
     "Doanh thu mất": fmt_eur(i3d["revenue_loss"], 0),
     "% tổng mất": f"{i3d['pct_total_loss']*100:.2f}%"},
    {"Tổ hợp": "Online TA×TA/TO + lead>60 + Jul-Aug (BR-REV-02)", "Booking": i3d["br_bookings"],
     "Tỷ lệ hủy": f"{i3d['br_cancel_rate']*100:.2f}%",
     "Doanh thu mất": fmt_eur(i3d["br_revenue_loss"], 0),
     "% tổng mất": f"{i3d['br_pct_total_loss']*100:.2f}%"},
])
summary_3d

## 2. Revenue Loss (Doanh thu tiềm năng mất do hủy)

`booking_revenue = ADR × total_nights` cho booking bị hủy — baseline cho **OB-01**.

In [ ]:
rev = analyze_revenue_loss(df)

print(f"Tổng doanh thu mất: {fmt_eur(rev['total_loss'], 0)}")
print(f"Tỷ lệ mất / tiềm năng: {rev['loss_pct_of_potential']*100:.2f}%")
print(f"Mean mất/đêm Jul-Aug: {fmt_eur(rev['peak_mean_loss_per_night'])}")

rev["by_month"][["sum", "count"]].dropna().assign(
    sum_M=lambda x: (x["sum"] / 1e6).round(2)
)[["count", "sum_M"]]

## 3. Room Mis-match — Upgrade vs Downgrade

Phân loại chuyển phòng theo thứ hạng `reserved → assigned`. **Free Upgrade proxy:** đặt phòng A/B nhưng nhận hạng cao hơn.

In [ ]:
mm = analyze_room_mismatch(df)

print(f"Tỷ lệ mis-match: {mm['mismatch_rate']*100:.2f}%")
print(f"Chênh ADR (khớp - không khớp): {fmt_eur(mm['adr_gap'])}")
print(f"Free upgrade proxy: {fmt_int(mm['free_upgrade_count'])} ({mm['free_upgrade_pct_of_mismatch']*100:.1f}% mis-match)")

mm["summary"].round(2)

## 4. Deposit Simulation

Kịch bản BRD: cọc 1 đêm cho **Online TA + lead_time > 30**. Giả định: hủy −30%, volume −10%.

In [ ]:
dep = analyze_deposit_simulation(df)

dep_table = pd.DataFrame([
    {"Kịch bản": "Hiện tại", "Booking": int(dep["bookings"]),
     "Tỷ lệ hủy": f"{dep['cancel_rate']*100:.2f}%",
     "Net Revenue": fmt_eur(dep["baseline_net"], 0)},
    {"Kịch bản": "Sau cọc (chỉ lưu trú)", "Booking": int(dep["scenario_bookings"]),
     "Tỷ lệ hủy": f"{dep['scenario_cancel_rate']*100:.2f}%",
     "Net Revenue": fmt_eur(dep["scenario_net"], 0)},
    {"Kịch bản": "Sau cọc (+ cọc giữ lại)", "Booking": int(dep["scenario_bookings"]),
     "Tỷ lệ hủy": f"{dep['scenario_cancel_rate']*100:.2f}%",
     "Net Revenue": fmt_eur(dep["scenario_net_with_deposit"], 0)},
])
dep_table

## 5. Xuất biểu đồ & báo cáo MD / BRD v1.2

In [ ]:
figs = save_figures(i3d, rev, mm, dep, df)
write_report(i3d, rev, mm, dep, figs)
write_brd(i3d, rev, mm, dep)

print("Đã ghi:")
print(" - reports/12_brd_gap_analysis.md")
print(" - reports/12_brd_v1_2.md")
print(f" - {len(figs)} biểu đồ trong reports/figures/12/")